# 🔥 Auto Fire Dept Reporting
**E-gle Eye 신뢰도 기반 재난 대응 서포트 시스템**
https://claude.ai/share/d5657cf6-b7e6-4fef-860b-81590fa02db2

---
### 동작 조건
- **Red 상태** + **신뢰도 90% 이상** + **5초 지속** → 자동 신고
- Building Location Excel 연동 (Fire_Dept_Phone / Fire_Dept_Email)
- SMS: Twilio 스텁 / Email: SMTP 스텁

### 파일 구조
```
project/
├── Auto_Fire_Dept_Reporting.ipynb  ← 현재 파일
├── config.json                      ← Twilio / SMTP 설정
└── building_location.xlsx           ← 건물별 소방서 연락처
```

## 📦 Cell 1: 라이브러리 설치 및 임포트

In [ ]:
# 필요 라이브러리 설치 (최초 1회)
# !pip install twilio openpyxl pandas requests

import json
import time
import smtplib
import logging
import threading
import pandas as pd
from datetime import datetime
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from pathlib import Path

# Twilio (설치된 경우 실제 사용, 없으면 스텁으로 동작)
try:
    from twilio.rest import Client as TwilioClient
    TWILIO_AVAILABLE = True
except ImportError:
    TWILIO_AVAILABLE = False
    print("[INFO] Twilio 미설치 → SMS 스텁 모드로 동작합니다.")

# 로거 설정
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler("fire_report.log", encoding="utf-8")
    ]
)
logger = logging.getLogger("AutoFireReport")
print("✅ 라이브러리 로드 완료")

## ⚙️ Cell 2: config.json 로드 및 초기 설정

In [ ]:
# ────────────────────────────────────────────────
# config.json 예시 (프로젝트 루트에 저장)
# ────────────────────────────────────────────────
# {
#   "twilio": {
#     "account_sid": "ACxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx",
#     "auth_token":  "your_auth_token",
#     "from_phone":  "+1234567890"
#   },
#   "smtp": {
#     "host":     "smtp.gmail.com",
#     "port":     587,
#     "user":     "your_email@gmail.com",
#     "password": "your_app_password",
#     "from_name": "E-gle Eye 화재감지시스템"
#   },
#   "building_excel": "building_location.xlsx",
#   "detection": {
#     "red_confidence_threshold": 0.90,
#     "sustain_seconds": 5
#   },
#   "stub_mode": true
# }
# ────────────────────────────────────────────────

CONFIG_PATH = "config.json"

def load_config(path: str) -> dict:
    """config.json 로드. 파일 없으면 기본 스텁 설정 반환."""
    if Path(path).exists():
        with open(path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        logger.info(f"config 로드 완료: {path}")
        return cfg
    else:
        logger.warning(f"{path} 없음 → 기본 스텁 설정 사용")
        return {
            "twilio": {"account_sid": "", "auth_token": "", "from_phone": "+00000000000"},
            "smtp":   {"host": "smtp.gmail.com", "port": 587, "user": "", "password": "", "from_name": "E-gle Eye"},
            "building_excel": "building_location.xlsx",
            "detection": {"red_confidence_threshold": 0.90, "sustain_seconds": 5},
            "stub_mode": True
        }

CONFIG = load_config(CONFIG_PATH)
STUB_MODE          = CONFIG.get("stub_mode", True)
CONF_THRESHOLD     = CONFIG["detection"]["red_confidence_threshold"]  # 0.90
SUSTAIN_SECONDS    = CONFIG["detection"]["sustain_seconds"]            # 5
BUILDING_EXCEL     = CONFIG["building_excel"]

print(f"✅ 설정 로드 | 스텁모드={STUB_MODE} | 임계값={CONF_THRESHOLD} | 지속시간={SUSTAIN_SECONDS}s")

## 🏢 Cell 3: Building Location Excel 연동

In [ ]:
# ────────────────────────────────────────────────
# building_location.xlsx 필수 컬럼:
#   Building_ID | Building_Name | Address
#   | Camera_ID | Fire_Dept_Phone | Fire_Dept_Email
# ────────────────────────────────────────────────

def load_building_info(excel_path: str) -> pd.DataFrame:
    """Building Location Excel 로드. 없으면 샘플 데이터 반환."""
    required_cols = ["Building_ID", "Building_Name", "Address",
                     "Camera_ID", "Fire_Dept_Phone", "Fire_Dept_Email"]
    
    if Path(excel_path).exists():
        df = pd.read_excel(excel_path, dtype=str)
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f"Excel 필수 컬럼 누락: {missing}")
        logger.info(f"Building Excel 로드: {excel_path} ({len(df)}개 건물)")
        return df
    else:
        logger.warning(f"{excel_path} 없음 → 샘플 데이터 사용")
        sample = pd.DataFrame([
            {"Building_ID": "B001", "Building_Name": "본관",
             "Address": "서울시 강남구 테헤란로 123",
             "Camera_ID": "CAM_01",
             "Fire_Dept_Phone": "+821012345678",
             "Fire_Dept_Email": "firestation@example.com"},
            {"Building_ID": "B002", "Building_Name": "별관",
             "Address": "서울시 강남구 테헤란로 125",
             "Camera_ID": "CAM_02",
             "Fire_Dept_Phone": "+821098765432",
             "Fire_Dept_Email": "fire2@example.com"},
        ])
        return sample

def get_contact_by_camera(df: pd.DataFrame, camera_id: str) -> dict | None:
    """Camera_ID로 건물 연락처 조회."""
    row = df[df["Camera_ID"] == camera_id]
    if row.empty:
        logger.error(f"Camera_ID '{camera_id}' 없음")
        return None
    r = row.iloc[0]
    return {
        "building_id":   r["Building_ID"],
        "building_name": r["Building_Name"],
        "address":       r["Address"],
        "phone":         r["Fire_Dept_Phone"],
        "email":         r["Fire_Dept_Email"],
    }

BUILDING_DF = load_building_info(BUILDING_EXCEL)
print("✅ Building 정보 로드")
BUILDING_DF

## 📱 Cell 4: SMS 발송 (Twilio / 스텁)

In [ ]:
def send_sms(to_phone: str, message: str, config: dict, stub: bool = True) -> bool:
    """
    SMS 발송 함수.
    stub=True  → 실제 발송 없이 로그만 출력 (개발/테스트용)
    stub=False → Twilio API 실제 호출
    """
    if stub or not TWILIO_AVAILABLE:
        logger.info(f"[SMS 스텁] TO={to_phone} | MSG={message}")
        print(f"\n📱 [SMS 스텁 발송]")
        print(f"   수신: {to_phone}")
        print(f"   내용: {message}")
        return True

    # ── 실제 Twilio 발송 ──
    try:
        tw_cfg = config["twilio"]
        client = TwilioClient(tw_cfg["account_sid"], tw_cfg["auth_token"])
        msg = client.messages.create(
            body=message,
            from_=tw_cfg["from_phone"],
            to=to_phone
        )
        logger.info(f"[SMS 발송 성공] SID={msg.sid} | TO={to_phone}")
        return True
    except Exception as e:
        logger.error(f"[SMS 발송 실패] {e}")
        return False


def build_sms_message(contact: dict, confidence: float, timestamp: str) -> str:
    """SMS 메시지 포맷 생성."""
    return (
        f"[화재감지-E-gle Eye] 🔥\n"
        f"건물: {contact['building_name']}\n"
        f"주소: {contact['address']}\n"
        f"신뢰도: {confidence*100:.1f}%\n"
        f"감지시각: {timestamp}\n"
        f"즉각 출동 요청"
    )

print("✅ SMS 함수 정의 완료")

## 📧 Cell 5: 이메일 발송 (SMTP / 스텁)

In [ ]:
def send_email(to_email: str, contact: dict, confidence: float,
               timestamp: str, config: dict, stub: bool = True) -> bool:
    """
    이메일 발송 함수.
    stub=True  → 실제 발송 없이 로그만 출력
    stub=False → SMTP 실제 발송
    """
    subject = f"[긴급-화재감지] {contact['building_name']} 화재 자동 신고"
    body_html = f"""
    <html><body>
    <h2 style='color:red;'>🔥 화재 자동 감지 알림 - E-gle Eye</h2>
    <table border='1' cellpadding='8' style='border-collapse:collapse;'>
      <tr><td><b>건물명</b></td><td>{contact['building_name']}</td></tr>
      <tr><td><b>건물 ID</b></td><td>{contact['building_id']}</td></tr>
      <tr><td><b>주소</b></td><td>{contact['address']}</td></tr>
      <tr><td><b>감지 신뢰도</b></td><td style='color:red;font-weight:bold;'>{confidence*100:.1f}%</td></tr>
      <tr><td><b>감지 상태</b></td><td style='color:red;'>🔴 RED</td></tr>
      <tr><td><b>감지 시각</b></td><td>{timestamp}</td></tr>
    </table>
    <br>
    <p style='color:red;'><b>골든타임 5분 내 즉각 출동을 요청드립니다.</b></p>
    <p style='font-size:12px;color:gray;'>본 메일은 E-gle Eye 화재감지시스템에 의해 자동 발송되었습니다.</p>
    </body></html>
    """

    if stub:
        logger.info(f"[EMAIL 스텁] TO={to_email} | SUBJECT={subject}")
        print(f"\n📧 [EMAIL 스텁 발송]")
        print(f"   수신: {to_email}")
        print(f"   제목: {subject}")
        print(f"   내용: {body_html[:200]}...")
        return True

    # ── 실제 SMTP 발송 ──
    try:
        smtp_cfg = config["smtp"]
        msg = MIMEMultipart("alternative")
        msg["Subject"] = subject
        msg["From"]    = f"{smtp_cfg['from_name']} <{smtp_cfg['user']}>"
        msg["To"]      = to_email
        msg.attach(MIMEText(body_html, "html", "utf-8"))

        with smtplib.SMTP(smtp_cfg["host"], smtp_cfg["port"]) as server:
            server.starttls()
            server.login(smtp_cfg["user"], smtp_cfg["password"])
            server.sendmail(smtp_cfg["user"], to_email, msg.as_string())

        logger.info(f"[EMAIL 발송 성공] TO={to_email}")
        return True
    except Exception as e:
        logger.error(f"[EMAIL 발송 실패] {e}")
        return False

print("✅ 이메일 함수 정의 완료")

## 🔴 Cell 6: 핵심 로직 - 신뢰도 기반 자동 신고 트리거

In [ ]:
class FireReportTrigger:
    """
    Red 상태 + 신뢰도 >= 90% 를 5초 이상 지속 시
    소방서에 SMS/Email 자동 발송.
    
    ─ 중복 신고 방지: 동일 카메라 신고 후 cooldown_seconds 동안 재신고 차단
    """

    def __init__(self, config: dict, building_df: pd.DataFrame,
                 threshold: float = 0.90, sustain: int = 5,
                 cooldown_seconds: int = 300):
        self.config      = config
        self.building_df = building_df
        self.threshold   = threshold       # 0.90
        self.sustain     = sustain         # 5초
        self.cooldown    = cooldown_seconds # 5분 재신고 방지
        self.stub_mode   = config.get("stub_mode", True)

        # {camera_id: {"start_time": float, "reported": bool, "last_report": float}}
        self._state: dict = {}
        self._lock = threading.Lock()

    # ── 외부에서 매 프레임/이벤트마다 호출 ──
    def update(self, camera_id: str, status: str, confidence: float) -> dict:
        """
        Parameters
        ----------
        camera_id  : str   예) 'CAM_01'
        status     : str   'RED' | 'YELLOW' | 'GREEN'
        confidence : float 0.0 ~ 1.0

        Returns
        -------
        dict  결과 요약
        """
        now = time.time()
        result = {"camera_id": camera_id, "action": "MONITORING",
                  "status": status, "confidence": confidence}

        with self._lock:
            state = self._state.setdefault(camera_id, {
                "start_time": None, "reported": False, "last_report": 0
            })

            # ── 조건 불충족 시 타이머 리셋 ──
            if status.upper() != "RED" or confidence < self.threshold:
                state["start_time"] = None
                state["reported"]   = False
                result["action"] = "RESET"
                return result

            # ── 조건 충족: 타이머 시작 ──
            if state["start_time"] is None:
                state["start_time"] = now
                logger.info(f"[{camera_id}] 🔴 Red+{confidence*100:.1f}% 감지 → 타이머 시작")
                result["action"] = "TIMER_START"
                return result

            elapsed = now - state["start_time"]
            result["elapsed"] = round(elapsed, 2)

            # ── 아직 지속 시간 미달 ──
            if elapsed < self.sustain:
                result["action"] = f"WAITING ({elapsed:.1f}s / {self.sustain}s)"
                return result

            # ── 쿨다운 중 (재신고 방지) ──
            if state["reported"] and (now - state["last_report"]) < self.cooldown:
                remain = self.cooldown - (now - state["last_report"])
                result["action"] = f"COOLDOWN ({remain:.0f}s 남음)"
                return result

            # ── 🔥 자동 신고 실행 ──
            self._report(camera_id, confidence, now)
            state["reported"]    = True
            state["last_report"] = now
            result["action"] = "REPORTED"
            return result

    def _report(self, camera_id: str, confidence: float, ts: float):
        """SMS + Email 동시 발송."""
        timestamp = datetime.fromtimestamp(ts).strftime("%Y-%m-%d %H:%M:%S")
        contact   = get_contact_by_camera(self.building_df, camera_id)

        if contact is None:
            logger.error(f"[{camera_id}] 연락처 없음 → 신고 불가")
            return

        logger.warning(
            f"🔥 자동 신고 실행 | 건물={contact['building_name']} "
            f"| 신뢰도={confidence*100:.1f}% | 시각={timestamp}"
        )

        # SMS 발송
        sms_msg = build_sms_message(contact, confidence, timestamp)
        sms_ok  = send_sms(contact["phone"], sms_msg, self.config, stub=self.stub_mode)

        # Email 발송
        email_ok = send_email(contact["email"], contact, confidence,
                              timestamp, self.config, stub=self.stub_mode)

        logger.info(f"신고 결과 | SMS={'✅' if sms_ok else '❌'} "
                    f"| Email={'✅' if email_ok else '❌'}")


# 인스턴스 생성
trigger = FireReportTrigger(
    config       = CONFIG,
    building_df  = BUILDING_DF,
    threshold    = CONF_THRESHOLD,
    sustain      = SUSTAIN_SECONDS,
    cooldown_seconds = 300
)
print("✅ FireReportTrigger 초기화 완료")

## 🧪 Cell 7: 시뮬레이션 테스트

In [ ]:
def simulate_detection_stream(camera_id: str, events: list, interval: float = 1.0):
    """
    감지 이벤트 스트림 시뮬레이션.
    events: [{"status": "RED", "confidence": 0.95}, ...]
    interval: 이벤트 간 딜레이(초)
    """
    print(f"\n{'='*60}")
    print(f"🎬 시뮬레이션 시작: {camera_id}")
    print(f"{'='*60}")

    for i, evt in enumerate(events):
        result = trigger.update(camera_id, evt["status"], evt["confidence"])
        print(f"  [{i+1:02d}] {evt['status']:6s} {evt['confidence']*100:5.1f}% "
              f"→ {result.get('action', '')}")
        if result.get("action") == "REPORTED":
            print(f"  ✅ 자동 신고 완료!")
            break
        time.sleep(interval)

    print(f"{'='*60}\n")


# ── 테스트 1: 정상 신고 흐름 (Red + 90%+ 5초 지속) ──
print("[테스트 1] Red 90%+ 5초 지속 → 자동 신고")
test_events_1 = [
    {"status": "RED", "confidence": 0.92},  # 타이머 시작
    {"status": "RED", "confidence": 0.94},  # 2초
    {"status": "RED", "confidence": 0.91},  # 3초
    {"status": "RED", "confidence": 0.96},  # 4초
    {"status": "RED", "confidence": 0.95},  # 5초 → 신고!
]
simulate_detection_stream("CAM_01", test_events_1, interval=1.0)

In [ ]:
# ── 테스트 2: 오탐지 방지 (Red 중 신뢰도 하락으로 리셋) ──
print("[테스트 2] Red 중 신뢰도 하락 → 타이머 리셋 (오탐 방지)")

# 트리거 상태 초기화 후 테스트
trigger._state.clear()

test_events_2 = [
    {"status": "RED",    "confidence": 0.92},  # 타이머 시작
    {"status": "RED",    "confidence": 0.93},
    {"status": "RED",    "confidence": 0.78},  # 신뢰도 하락 → 리셋
    {"status": "RED",    "confidence": 0.85},
    {"status": "YELLOW", "confidence": 0.60},  # 상태 변화 → 리셋
]
simulate_detection_stream("CAM_02", test_events_2, interval=1.0)
print("→ 5초 미만으로 신고 발생 안 함 (정상)")

In [ ]:
# ── 테스트 3: 정밀 경계값 테스트 ──
print("[테스트 3] 신뢰도 89.9% → 신고 안 됨")
trigger._state.clear()

test_events_3 = [
    {"status": "RED", "confidence": 0.899},
    {"status": "RED", "confidence": 0.889},
    {"status": "RED", "confidence": 0.879},
    {"status": "RED", "confidence": 0.899},
    {"status": "RED", "confidence": 0.895},
    {"status": "RED", "confidence": 0.898},
]
simulate_detection_stream("CAM_01", test_events_3, interval=1.0)
print("→ 90% 미만 → 신고 없음 (정상)")

## 🔌 Cell 8: 외부 시스템 연동 인터페이스

기존 E-gle Eye 메인 파이프라인에서 아래 함수를 호출하면 됩니다.

In [ ]:
# ────────────────────────────────────────────────────────────
# 외부 연동용 공개 인터페이스
# 기존 파이프라인에서 아래처럼 호출하세요:
#
#   from Auto_Fire_Dept_Reporting import process_detection_event
#
#   result = process_detection_event(
#       camera_id  = "CAM_01",
#       status     = "RED",        # 'RED' | 'YELLOW' | 'GREEN'
#       confidence = 0.95          # 0.0 ~ 1.0
#   )
# ────────────────────────────────────────────────────────────

def process_detection_event(camera_id: str, status: str, confidence: float) -> dict:
    """
    E-gle Eye 메인 파이프라인 연동 진입점.
    매 감지 프레임마다 호출.

    Returns
    -------
    dict:
        camera_id  : str
        action     : str  'MONITORING'|'TIMER_START'|'WAITING'|'REPORTED'|'COOLDOWN'|'RESET'
        status     : str
        confidence : float
        elapsed    : float (있을 경우)
    """
    return trigger.update(camera_id, status, confidence)


print("✅ 외부 연동 인터페이스 준비 완료")
print()
print("사용 예시:")
print("  result = process_detection_event('CAM_01', 'RED', 0.95)")
print("  print(result)")
print()

# 간단 동작 확인
trigger._state.clear()
sample_result = process_detection_event("CAM_01", "RED", 0.95)
print("샘플 결과:", sample_result)

## 📋 Cell 9: 설정 요약 및 체크리스트

In [ ]:
print("="*60)
print("  E-gle Eye Auto Fire Dept Reporting - 설정 요약")
print("="*60)
print(f"  스텁 모드        : {'✅ ON (실제 발송 안 함)' if STUB_MODE else '🔴 OFF (실제 발송 활성화)'}")
print(f"  신뢰도 임계값    : {CONF_THRESHOLD*100:.0f}%")
print(f"  지속 시간        : {SUSTAIN_SECONDS}초")
print(f"  재신고 쿨다운    : 300초 (5분)")
print(f"  Building Excel  : {BUILDING_EXCEL}")
print(f"  Twilio 사용 가능 : {'✅' if TWILIO_AVAILABLE else '❌ (pip install twilio 필요)'}")
print()
print("  [체크리스트]")
print(f"  {'✅' if Path(CONFIG_PATH).exists() else '❌'} config.json 존재")
print(f"  {'✅' if Path(BUILDING_EXCEL).exists() else '⚠️ 샘플 사용 중'} building_location.xlsx 존재")
print(f"  {'✅' if TWILIO_AVAILABLE else '⚠️ 스텁 모드'} Twilio 설치")
print()
print("  실제 운영 전 config.json의 stub_mode를 false로 변경하세요.")
print("="*60)